# 3. 조건부 해결 경로

**시나리오:** 일반 billing 문의는 담당 queue로, 전사 장애는 incident escalation으로 보냅니다.

**학습 목표:** `add_conditional_edges`가 state의 route 값에 따라 실제 graph node를 다르게 실행하는 방식을 관찰합니다.

## 중요 변수·함수

- `run_ticket_workflow()`: 분류·필터 검색·계획·routing을 실행하는 canonical 함수입니다.
- `route`: `billing_queue` 또는 `incident_escalation` 같은 소유권 결과입니다.
- `steps`: 선택된 queue node가 실제로 실행됐는지 보여줍니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 서로 다른 입력 두 개로 graph 경로를 비교합니다.
from week2.app import TicketRequest, create_fixture_services, run_ticket_workflow

services = create_fixture_services()
billing = run_ticket_workflow(TicketRequest(subject='Duplicate invoice', description='Charged twice', customer_tier='standard'), services)
outage = run_ticket_workflow(TicketRequest(subject='Service outage', description='All users are blocked', customer_tier='enterprise'), services)

In [ ]:
# 마지막 node가 계산 값이 아니라 실제 조건부 edge의 목적지인지 확인합니다.
assert billing['route'] == billing['steps'][-1] == 'billing_queue'
assert outage['route'] == outage['steps'][-1] == 'incident_escalation'
{'billing_steps': billing['steps'], 'outage_steps': outage['steps']}

## 예측 과제와 해석

**예측 과제:** priority가 urgent이면 category가 billing이어도 어느 경로가 우선해야 할지 설명하세요.

**해석:** route 문자열만 계산하는 것과 조건부 edge로 다른 node를 실행하는 것은 다릅니다. trace가 분기를 증명해야 합니다.